# 🛠️ 00 环境验证 Notebook

**目的**：确认 AMD Radeon GPU + ROCm + PyTorch + Diffusers 链路通畅。

**预计耗时**：3–8 分钟（首次会从 HuggingFace 下载 `stabilityai/sd-turbo` 约 2GB）。

**通过标志**：最后一个 cell 输出 `🎉 ALL CHECKS PASSED` 并在 `outputs/00_env_check.png` 生成一张图。

如果这一关跑通了，后面所有功能（文生图 / 图生视频 / 风格化 / 一键短片）都基于此环境。

## 1️⃣ 系统信息

In [ ]:
import sys, platform
print(f"Python: {sys.version.split()[0]}")
print(f"OS:     {platform.system()} {platform.release()}")
print(f"Arch:   {platform.machine()}")

## 2️⃣ ROCm + GPU 检查

In [ ]:
import torch

print(f"PyTorch:  {torch.__version__}")
print(f"ROCm/HIP: {torch.version.hip}")
print(f"CUDA alias available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ GPU 不可用。常见原因：\n"
        "  1. ROCm 驱动未装 / 版本不对\n"
        "  2. PyTorch 装的是 CPU 版，没装 ROCm 版\n"
        "  3. 当前用户没权限访问 GPU（需加入 video 或 render 组）\n"
        "解决：参考 README 重新装 ROCm 版 PyTorch。"
    )

device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)
print(f"✅ Device:  {torch.cuda.get_device_name(device)}")
print(f"   VRAM:    {props.total_memory / 1e9:.1f} GB")
print(f"   Compute: {props.major}.{props.minor}")

## 3️⃣ 必要依赖检查

In [ ]:
import importlib

required = {
    "diffusers":   "Diffusers",
    "transformers":"Transformers",
    "accelerate":  "Accelerate",
    "safetensors": "Safetensors",
    "PIL":         "Pillow",
    "cv2":         "OpenCV",
    "imageio":     "ImageIO",
    "av":          "PyAV",
}

missing = []
for mod, name in required.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        print(f"  ✅ {name:12s} {ver}")
    except ImportError:
        print(f"  ❌ {name:12s} NOT INSTALLED")
        missing.append(mod)

if missing:
    print(f"\n⚠️ 缺失依赖: {missing}")
print("   建议运行: pip install -r requirements.txt")
if missing:
    raise ImportError(f"请先安装缺失依赖: {missing}")

## 4️⃣ GPU 算力小测（FP16 matmul）

In [ ]:
import torch, time

size = 4096
x = torch.randn(size, size, device="cuda", dtype=torch.float16)
y = torch.randn(size, size, device="cuda", dtype=torch.float16)

# warmup
_ = torch.matmul(x, y)
torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(10):
    z = torch.matmul(x, y)
torch.cuda.synchronize()
dt = (time.perf_counter() - t0) / 10
tflops = (2 * size ** 3) / dt / 1e12

print(f"✅ 4096x4096 FP16 matmul: {dt*1000:.1f} ms/iter, ~{tflops:.1f} TFLOPS")
del x, y, z
torch.cuda.empty_cache()

## 5️⃣ 文生图实测（SD-Turbo，1–2 步即可）

第一次会从 HuggingFace 下载 `stabilityai/sd-turbo`（约 2GB），请耐心等待。

如果 VM 在国内拉 HF 慢，可设置镜像：
```python
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
```

In [ ]:
# 可选：国内镜像加速
import os
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from diffusers import AutoPipelineForText2Image
import torch, time

print("Loading SD-Turbo ...")
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

# 第一次 warmup（首次推理会做编译/缓存）
_ = pipe("warmup", num_inference_steps=1, guidance_scale=0.0).images[0]
torch.cuda.synchronize()

t0 = time.perf_counter()
image = pipe(
    prompt="a cute robot reading a book on the moon, anime style, vibrant colors",
    num_inference_steps=2,
    guidance_scale=0.0,
).images[0]
torch.cuda.synchronize()
dt = time.perf_counter() - t0

print(f"✅ Generated in {dt:.1f}s")
print(f"   Image size: {image.size}")

In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)
out = "../outputs/00_env_check.png"
image.save(out)
print(f"✅ Saved: {out}")
print(f"   File size: {os.path.getsize(out) / 1024:.1f} KB")

alloc    = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved()  / 1e9
total    = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n📊 VRAM: allocated={alloc:.2f}G, reserved={reserved:.2f}G, total={total:.1f}G")
print(f"   Free:  {total - reserved:.2f} GB")

print("\n" + "=" * 50)
print("🎉 ALL CHECKS PASSED — 环境就绪！")
print("=" * 50)
print("\n下一步：打开 01_text2image.ipynb")